## 從 XXX 抓取 所有測站的名稱，並且分稱花蓮區域&台北區域

In [5]:
import re


# 讀取文字檔 (請確保檔案編碼正確，通常為 utf-8)
with open('../source/測站資料與位置.txt', 'r', encoding='utf-8') as file:
    content = file.read()
# 使用正規表達式找出所有 'Station:' 後面的一組代碼
station_codes = re.findall(r'Station:\s+([A-Z0-9]+)', content)

# 印出結果
print(station_codes)

['ZUZH', 'TWA', 'TAP', 'NHY', 'ANP', 'TWY', 'TWS1', 'TWB1', 'TIPB', 'NWR', 'NWL', 'NWF', 'NTS', 'NSM', 'NHDH', 'BAC', 'TWF1', 'TWD', 'SHUL', 'HWA', 'FULB', 'EYUL', 'EYL', 'ETM', 'ETLH', 'ETL', 'ESL', 'EHYH', 'EHY', 'EHP', 'EGFH', 'EGC']


In [ ]:
station_code = {
    'Hualien': [
        'SHUL', 'HWA', 'EYUL', 'EYL', 'ETM', 
        'ETLH', 'ESL', 'EHYH', 'EHP', 'EGFH', 'EGC'
    ],
    'Taipei': [
        'ZUZH', 'TAP', 'ANP', 'TWS1', 'TWB1', 
        'TIPB', 'NWR', 'NWL', 'NWF', 'NTS', 'NSM', 'NHDH', 'BAC'
    ]
}

""{
    "Hualien": [
        "TWF1", "TWD", "SHUL", "HWA", "FULB", "EYUL", "EYL", "ETM", 
        "ETLH", "ETL", "ESL", "EHYH", "EHY", "EHP", "EGFH", "EGC"
    ],
    "Taipei": [
        "ZUZH", "TWA", "TAP", "NHY", "ANP", "TWY", "TWS1", "TWB1", 
        "TIPB", "NWR", "NWL", "NWF", "NTS", "NSM", "NHDH", "BAC"
    ]
}

{
    "event_id":{
        "Hualien":{
            "TWF1":{
            "waveform":[-5到+15的值list]},
            "TWD":{
            "waveform":-5到+15的值},
            ...
        },
        "Taipei":{
            "ZUZH":{
                "pga":數值
            }
        },
        "first-p-arrival_sample":數值
    },


}

In [10]:
import pandas as pd

# 1. 準備你的測站字典
station_dict = {
    "Hualien": [
        "TWF1", "TWD", "SHUL", "HWA", "FULB", "EYUL", "EYL", "ETM", 
        "ETLH", "ETL", "ESL", "EHYH", "EHY", "EHP", "EGFH", "EGC"
    ],
    "Taipei": [
        "ZUZH", "TWA", "TAP", "NHY", "ANP", "TWY", "TWS1", "TWB1", 
        "TIPB", "NWR", "NWL", "NWF", "NTS", "NSM", "NHDH", "BAC"
    ]
}

# 將花蓮與台北的測站合併成一個大 List
target_stations = station_dict["Hualien"] + station_dict["Taipei"]

# 2. 讀取原始的 CSV 檔案
df = pd.read_csv('../source/all_metadata.csv') 

# ----------------- 開始執行核心流程 -----------------

# 步驟一：抓取所有不同的 event_id 
unique_event_ids = df['source_event_id'].unique()
filtered_results = []

# 步驟二：依照 event_id 依序去過濾 station_code
for event in unique_event_ids:
    event_data = df[df['source_event_id'] == event]
    
    # 使用合併後的 target_stations 一次性過濾
    target_data = event_data[event_data['station_code'].isin(target_stations)]
    
    # 如果這個事件過濾後有資料，就加入結果中
    if not target_data.empty:
        filtered_results.append(target_data)


    # 將 'source_event_id' 放到清單的最前面
    cols = final_df.columns.tolist()
    cols.remove('source_event_id')
    cols = ['source_event_id'] + cols
    final_df = final_df[cols]

# 步驟三：將結果存成一個新的 CSV 檔案
if len(filtered_results) > 0:
    final_df = pd.concat(filtered_results, ignore_index=True)
    
    
    output_filename = '../source/filtered_hualien_taipei_events.csv'
    final_df.to_csv(output_filename, index=False)
    
    print(f"處理完成！總共處理了 {len(unique_event_ids)} 個獨立事件。")
    print(f"篩選後的結果已儲存為 '{output_filename}'，共包含 {len(final_df)} 筆紀錄。")
else:
    print("這些事件中，沒有任何一個符合花蓮或台北測站的紀錄。")

# 步驟四:# 只要這一行，就能完成分群、找最小值、轉字典的所有動作！
event_earliest_p_arrival_sample = df.groupby('source_event_id')['trace_p_arrival_sample'].min().to_dict()
print(event_earliest_p_arrival_sample)

處理完成！總共處理了 5780 個獨立事件。
篩選後的結果已儲存為 '../source/filtered_hualien_taipei_events.csv'，共包含 98532 筆紀錄。
{12010102300: 1884, 12010110450: 4108, 12010114230: 2264, 12010403340: 4244, 12010406530: 4847, 12010406560: 2921, 12010406590: 2909, 12010409160: 3557, 12010413210: 3032, 12010413240: 2954, 12010414410: 1643, 12010418360: 4725, 12010501480: 1300, 12010502290: 6402, 12010515450: 2482, 12010516320: 1642, 12010621400: 3303, 12010903400: 3704, 12010904250: 7001, 12010904280: 4257, 12010906160: 4115, 12010907060: 5388, 12010909330: 3884, 12010913260: 1193, 12011001250: 7480, 12011007400: 7518, 12011008000: 2526, 12011009340: 1632, 12011012440: 2858, 12011012530: 3465, 12011017410: 2542, 12011017420: 1296, 12011018210: 2236, 12011020550: 1830, 12011102160: 6819, 12011110020: 3399, 12011110450: 3608, 12011118110: 2032, 12011213220: 3642, 12011213440: 4064, 12011303120: 4663, 12011304400: 7317, 12011304430: 938, 12011304520: 9090, 12011310400: 1197, 12011314060: 1909, 12011318460: 853, 12011408160: